In [1]:
!nvidia-smi

Thu Jan  8 13:32:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:00:03.0 Off |                    0 |
| N/A   58C    P0             29W /   72W |    6665MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [3]:
from kernels.gla.triton.fwd_kernel import parallel_gla_kernel

In [4]:
import torch
import triton
import triton.language as tl

In [5]:
def triton_gla_forward(q, k, v):
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()

    B, L, H, D_QK = q.shape
    _, _, _, D_V = v.shape

    output = torch.empty_like(v)

    BLOCK_M = 16
    BLOCK_N = 16

    grid = (triton.cdiv(L, BLOCK_M), B * H)

    parallel_gla_kernel[grid](
        q, k, v, output,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        output.stride(0), output.stride(1), output.stride(2), output.stride(3),
        B, L, H,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        D_QK=D_QK,
        D_V=D_V,
    )
    return output

In [6]:
B, L, H, D = 4, 4096, 8, 128
D_QK = 2 * D
dtype = torch.float16
device = "cuda"

print(f"Benchmarking Triton Kernel...")
print(f"Config: Batch={B}, Len={L}, Heads={H}, Dim={D}, Dtype={dtype}")

q_hat = torch.randn((B, L, H, D_QK), device=device, dtype=dtype)
k_hat = torch.randn((B, L, H, D_QK), device=device, dtype=dtype)
v     = torch.randn((B, L, H, D),    device=device, dtype=dtype)

print("Warming up GPU and compiling kernel...")
for _ in range(5):
    _ = triton_gla_forward(q_hat, k_hat, v)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

print("Starting benchmark loop...")
start_event.record()

n_loops = 100
for _ in range(n_loops):
    _ = triton_gla_forward(q_hat, k_hat, v)

end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops

print(f"Total time ({n_loops} runs): {elapsed_time_ms:.2f} ms")
print(f"Average time per run:      {avg_time:.4f} ms")

Benchmarking Triton Kernel...
Config: Batch=4, Len=4096, Heads=8, Dim=128, Dtype=torch.float16
Warming up GPU and compiling kernel...


CompilationError: at 36:4:
    Q_ptr = Q + (i_b * stride_qb + i_h * stride_qh)
    K_ptr = K + (i_b * stride_kb + i_h * stride_kh)
    V_ptr = V + (i_b * stride_vb + i_h * stride_vh)
    Out_ptr = Out + (i_b * stride_ob + i_h * stride_oh)

    q_ptrs = Q_ptr + (offs_m[:, None] * stride_ql + offs_d_qk[None, :] * stride_qd)
    q = tl.load(q_ptrs, mask=offs_m[:, None] < L, other=0.0)

    acc = tl.zeros([BLOCK_M, D_V], dtype=tl.float16)
    loop_end = (pid_m + 1) * BLOCK_M

    for start_n in range(0, loop_end, BLOCK_N):
    ^
AssertionError('Loop-carried variable acc has initial type <[16, 128], fp16> but is re-assigned to <[16, 128], fp32> in loop! Please make sure that the type stays consistent.')